# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant Schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Note: Access attributes, not subscripting

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, using entity `@id`s as references.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']}: {field.get('name', '')} ({field.get('data_type', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We'll use the record set and field `@id`s identified above.

In [ ]:
# List of record set @ids to extract (Update as needed based on output of previous cell)
# Example record set ids:
# Suppose from above, the main table is cr:ProteinTable, referencing by @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Error loading records for RecordSet '@id': {record_set_id}. Error: {e}")

# For this notebook, we select the first record set as example
main_record_set_id = record_set_ids[0]
print(f"Using record set: {main_record_set_id}")
print("Sample records:")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic EDA: filtering, normalization, and grouping.

**Notes:**
- Choose a numeric field (e.g. 'Coverage_Percent', or 'MW_kDa') using the available columns from above.
- All columns are referenced by their field `@id` string.

In [ ]:
# Choose a numeric field for analysis, identified by its @id
# Update field @id below. E.g., '@id': 'cr:MW_kDa' or similar
numeric_field = None  # Set below
group_field = None    # Set below

# Find a numeric field in the loaded DataFrame
for col in dataframes[main_record_set_id].columns:
    if str(dataframes[main_record_set_id][col].dtype) in ['float64', 'int64']:
        numeric_field = col
        break

if numeric_field is None:
    # Try to find a likely numeric field by pattern
    for col in dataframes[main_record_set_id].columns:
        if any(w in col.lower() for w in ['percent', 'mw', 'count', 'coverage']):
            numeric_field = col
            break

print(f"Selected numeric field for analysis: {numeric_field}")

# Demonstrate a simple filter and normalization
df = dataframes[main_record_set_id].copy()

# Convert field to numeric if not already
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > 10]

print(f"Filtered {len(filtered_df)} records with {numeric_field} > 10:\n", filtered_df.head())
# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a categorical/group field for grouping
for col in df.columns:
    if col != numeric_field:
        nunique = df[col].nunique()
        if 1 < nunique < min(10, len(df) // 5):
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt

# Visualize histogram of numeric field
plt.figure(figsize=(8,4))
filtered_df[numeric_field].hist(bins=30, alpha=0.7)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, plot boxplot
if group_field:
    plt.figure(figsize=(10,5))
    filtered_df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.suptitle('')
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.
- We enumerated all record sets, their fields (`@id`), and loaded records as DataFrames.
- Basic EDA and visualization provided initial insights into numeric field distributions and relationships.
- For deeper analysis, explore additional fields, advanced domain-specific processing, or integrate with other omics resources.